# Qudor AI v2 — T4 fast + strong training
Запускайте сверху вниз. Все checkpoints и метрики сохраняются на Drive, повторный запуск продолжает обучение.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Выберите Runtime → Change runtime type → T4 GPU'
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Qudor
!python -m pip install -q -e .

In [ ]:
!python scripts/benchmark.py

## 1. Перенос опыта старых моделей
Старые replay-позиции используются для pretrain новой residual-сети.

In [ ]:
!python -m quoridor_ai.legacy_pretrain --legacy legacy --output /content/drive/MyDrive/Qudor_runs/legacy_pretrained.pt --epochs 5 --batch 1024

## 2. Быстрая проверка batched self-play

In [ ]:
!python -m quoridor_ai.train --config configs/colab_t4_fast.json --output /content/drive/MyDrive/Qudor_runs/v2 --init /content/drive/MyDrive/Qudor_runs/legacy_pretrained.pt

## 3. Длительный быстрый pretrain с автоматическим resume

In [ ]:
!python -m quoridor_ai.train --config configs/colab_t4_balanced.json --output /content/drive/MyDrive/Qudor_runs/v2

## 4. Сильный MCTS-finetune
Запускайте после накопления pretrain checkpoint.

In [ ]:
!python -m quoridor_ai.mcts_finetune --config configs/colab_t4_balanced.json --checkpoint /content/drive/MyDrive/Qudor_runs/v2/latest.pt --output /content/drive/MyDrive/Qudor_runs/v2/mcts_latest.pt --games 16 --sims 96

## Статистика

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
p='/content/drive/MyDrive/Qudor_runs/v2/metrics.csv'
df=pd.read_csv(p); display(df.tail(30))
df.plot(x='iteration',y=['total_loss','games_per_sec','positions_per_sec','vram_mb'],subplots=True,figsize=(13,13),grid=True); plt.show()